# Exploratory analysis of an instance

This notebook only *looks at* data; all logic lives in `src/railway_rostering`. The default instance is the synthetic `medium` preset (**synthetic data, not OptiFlux operational data**). Change `INSTANCE` to inspect another directory.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from railway_rostering import load_instance, solve_instance, ModelConfig, SolverConfig

INSTANCE = ROOT / "data/generated/medium"
inst = load_instance(INSTANCE)
print(inst.name, len(inst.crews), "crew,", len(inst.duties), "duties,", inst.total_open_demand, "open demand units")
print("synthetic:", inst.metadata.get("synthetic"), "| seed:", inst.metadata.get("seed"))

## Duties by activity and start hour

In [ ]:
duties = pd.DataFrame([{
    "duty_id": d.duty_id, "activity": d.activity, "start": d.start, "hour": d.start.hour,
    "date": d.start.date(), "route": f"{d.start_location}->{d.end_location}", "demand": d.demand,
    "existing": len(d.assigned_crew), "eligible_listed": len(d.eligible_crew or ()),
} for d in inst.duties])
display(duties.groupby(["activity", "route"]).agg(duties=("duty_id", "count"), demand=("demand", "sum")).sort_values("duties", ascending=False))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
duties["hour"].value_counts().sort_index().plot.bar(ax=ax[0], title="Duty start hour")
duties["date"].value_counts().sort_index().plot.bar(ax=ax[1], title="Duties per start date")
plt.tight_layout()

## Crew: types, bases, initial locations

In [ ]:
crew = pd.DataFrame([{
    "crew_id": c.crew_id, "type": c.crew_type.value, "base": c.base_location, "initial": c.initial_location,
    "rostered": c.rostered_start.strftime("%H:%M") if c.rostered_start else None,
    "prior_duty": c.prior_duty_end is not None, "next_commitment": c.next_commitment_start is not None,
    "n_quals": len(inst.qualifications.get(c.crew_id, ())) if inst.qualifications else None,
} for c in inst.crews.values()])
display(pd.crosstab(crew["base"], crew["type"]))
display(pd.crosstab(crew["type"], crew["rostered"]))
print("crew starting away from base:", int((crew["base"] != crew["initial"]).sum()))

## How many crew survive each candidate filter?

The funnel below comes from candidate generation's per-duty diagnostics (eligible -> available on that date -> allowed start time -> reachable in a legal sequence). This is how the outputs explain uncovered duties.

In [ ]:
from railway_rostering.candidate_generation import generate_candidates
cands = generate_candidates(inst, ModelConfig())
diag = pd.DataFrame([{
    "duty_id": k, "eligible": v.n_eligible, "available": v.n_available, "time_ok": v.n_time_ok,
    "feasible": v.n_reachable, "reason": v.structural_reason() or "",
} for k, v in cands.diagnostics.items()])
display(diag[["eligible", "available", "time_ok", "feasible"]].describe().round(1))
diag[["eligible", "available", "time_ok", "feasible"]].mean().plot.bar(title="Mean crew per duty after each filter")
print(cands.stats)

In [ ]:
display(diag[diag["reason"] != ""].merge(duties, on="duty_id")[["duty_id", "activity", "start", "route", "eligible", "available", "time_ok", "feasible", "reason"]])